# Semana 7 — Pipeline USD/PEN



#  Visualizaciones EDA

En esta sección se generan gráficos exploratorios para entender el comportamiento temporal del dataset.

Se visualiza:

- cantidad de noticias por día
- proporción diaria de sentimiento positivo, negativo y neutral
- distribución del índice diario de sentimiento

Estos gráficos permiten identificar patrones temporales, picos de noticias, predominancia de sentimiento neutral y posibles eventos extremos.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_daily["date"], df_daily["n_news_total"], marker="o")
plt.title("Noticias por día")
plt.xlabel("Fecha")
plt.ylabel("Cantidad de noticias")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGS_DIR / "week5_news_per_day.png", dpi=150)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(df_daily["date"], df_daily["share_pos"], marker="o", label="share_pos")
plt.plot(df_daily["date"], df_daily["share_neg"], marker="o", label="share_neg")
plt.plot(df_daily["date"], df_daily["share_neu"], marker="o", label="share_neu")
plt.title("Proporción diaria de sentimiento")
plt.xlabel("Fecha")
plt.ylabel("Proporción")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGS_DIR / "week5_daily_sentiment_shares.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(df_daily["sent_index_mean"].dropna(), bins=20)
plt.title("Distribución del índice diario de sentimiento")
plt.xlabel("sent_index_mean")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.savefig(FIGS_DIR / "week5_dist_sent_index_mean.png", dpi=150)
plt.show()

# 1. Métrica central, checklist y configuración

In [ ]:
# =========================
# 
# Validación 
# =========================

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
    precision_recall_curve,
    classification_report
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from datetime import datetime

CENTRAL_METRIC = "pr_auc"

print("Métrica central elegida:", CENTRAL_METRIC)
print("Justificación: PR-AUC se usa porque el target UP/DOWN está desbalanceado.")
print("Seed fija:", RANDOM_SEED)
print("Train:", train_df["date"].min(), "→", train_df["date"].max(), "| filas:", len(train_df))
print("Test :", test_df["date"].min(), "→", test_df["date"].max(), "| filas:", len(test_df))

# 3. Función de evaluación

In [ ]:
# =========================
# Función de evaluación
# =========================

def evaluate_model_week6(model_name, features, train_df, test_df):
    X_train = train_df[features].copy()
    X_test = test_df[features].copy()

    y_train_local = train_df["target_up"].astype(int).copy()
    y_test_local = test_df["target_up"].astype(int).copy()

    model = make_logreg()

    start = time.time()
    model.fit(X_train, y_train_local)
    elapsed = time.time() - start

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    metrics = {
        "model": model_name,
        "features": ", ".join(features),
        "accuracy": accuracy_score(y_test_local, y_pred),
        "f1_binary": f1_score(y_test_local, y_pred, zero_division=0),
        "f1_weighted": f1_score(y_test_local, y_pred, average="weighted", zero_division=0),
        "precision": precision_score(y_test_local, y_pred, zero_division=0),
        "recall": recall_score(y_test_local, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_test_local, y_score),
        "elapsed_seconds": elapsed,
        "n_train": len(y_train_local),
        "n_test": len(y_test_local),
        "n_features": len(features)
    }

    return metrics, y_pred, y_score, model

# 4. Experimentos A/B finales


Se realizaron experimentos A/B modificando una sola parte del modelo en cada variante para evaluar el impacto de nuevas features temporales.

### Baseline_lag1
Utiliza únicamente el sentimiento del día anterior (`lag_sent_1`).

### Var1_lag1_lag2
Agrega una segunda memoria temporal (`lag_sent_2`) para evaluar si la información de dos días previos mejora la capacidad predictiva.

### Var2_lags_volume_shares
Incorpora variables relacionadas con:

- volumen de noticias (`n_news_total`)
- proporción positiva (`share_pos`)
- proporción negativa (`share_neg`)
- proporción neutral (`share_neu`)

El objetivo es evaluar si la intensidad informativa y la composición del sentimiento aportan información adicional.

### Var3_full_temporal
Agrega además:

- promedio móvil de sentimiento (`rolling_sent_3`)
- volatilidad del sentimiento (`volatility_sent_3`)

para capturar tendencia e inestabilidad temporal.

In [ ]:
# =========================
# Experimentos A/B Semana 6
# =========================

experiments_week6 = [
    {
        "model": "Baseline_lag1",
        "features": ["lag_sent_1"],
        "change": "Referencia: sentimiento del día anterior.",
        "why": "Evalúa si el sentimiento rezagado contiene señal básica."
    },
    {
        "model": "Var1_lag1_lag2",
        "features": ["lag_sent_1", "lag_sent_2"],
        "change": "Agrega lag_sent_2.",
        "why": "Evalúa memoria temporal de dos días."
    },
    {
        "model": "Var2_lags_volume_shares",
        "features": [
            "lag_sent_1",
            "lag_sent_2",
            "n_news_total",
            "share_pos",
            "share_neg",
            "share_neu"
        ],
        "change": "Agrega volumen y composición de sentimiento.",
        "why": "Evalúa si la intensidad informativa y distribución positiva/negativa aportan señal."
    },
    {
        "model": "Var3_full_temporal",
        "features": [
            "lag_sent_1",
            "lag_sent_2",
            "rolling_sent_3",
            "volatility_sent_3",
            "n_news_total",
            "share_pos",
            "share_neg",
            "share_neu"
        ],
        "change": "Agrega rolling y volatilidad temporal.",
        "why": "Evalúa si la tendencia y variabilidad del sentimiento mejoran el desempeño."
    }
]

week6_results = []
week6_scores = {}
week6_predictions = {}

for exp in experiments_week6:
    metrics, y_pred, y_score, fitted = evaluate_model_week6(
        exp["model"],
        exp["features"],
        train_df,
        test_df
    )

    metrics["change"] = exp["change"]
    metrics["why"] = exp["why"]

    week6_results.append(metrics)
    week6_scores[exp["model"]] = y_score
    week6_predictions[exp["model"]] = y_pred

week6_results_df = pd.DataFrame(week6_results)

week6_results_df = week6_results_df[
    [
        "model",
        "features",
        "accuracy",
        "f1_binary",
        "f1_weighted",
        "precision",
        "recall",
        "pr_auc",
        "elapsed_seconds",
        "n_features",
        "n_train",
        "n_test",
        "change",
        "why"
    ]
]

display(week6_results_df)

week6_results_df.to_csv(
    LOGS_DIR / "week6_resultados_comparables.csv",
    index=False,
    encoding="utf-8-sig"
)

### Interpretación de resultados

La métrica central seleccionada para esta evaluación fue PR-AUC debido al desbalance existente entre las clases UP y DOWN.

Los resultados muestran que:

- Baseline_lag1 obtuvo un PR-AUC de 0.777.
- Var1_lag1_lag2 alcanzó el mejor desempeño con un PR-AUC de 0.786.
- Var2_lags_volume_shares obtuvo un PR-AUC de 0.746.
- Var3_full_temporal obtuvo un PR-AUC de 0.741.

La incorporación de `lag_sent_2` produjo una mejora ligera pero consistente respecto al baseline, sugiriendo que existe memoria temporal de corto plazo en la relación entre sentimiento financiero y movimientos del USD/PEN.

Por otro lado, la incorporación de variables adicionales de volumen, composición y volatilidad no produjo mejoras en este conjunto de datos, posiblemente debido al tamaño limitado del dataset o a ruido introducido por variables menos informativas.

Por esta razón, la variante Var1_lag1_lag2 fue seleccionada como la mejor candidata para las siguientes etapas de validación.

# 5. Gráfico PR-AUC + F1

### Interpretación de resultados A/B

La comparación se realizó utilizando dos métricas:

- **PR-AUC (métrica central):** mide la capacidad del modelo para distinguir correctamente las clases UP y DOWN en presencia de desbalance.
- **F1 Binary (métrica secundaria):** balance entre precision y recall para la clase positiva.

#### Resultados observados

| Modelo | PR-AUC | F1 Binary |
|---------|---------|---------|
| Baseline_lag1 | 0.777 | 0.563 |
| Var1_lag1_lag2 | **0.786** | 0.533 |
| Var2_lags_volume_shares | 0.746 | 0.500 |
| Var3_full_temporal | 0.741 | 0.545 |

#### Análisis

La variante **Var1_lag1_lag2** obtuvo el mejor desempeño según la métrica central (**PR-AUC = 0.786**), superando al baseline (**0.777**).

Este resultado sugiere que incorporar información de sentimiento de los dos días previos permite capturar mejor la dinámica temporal entre las noticias financieras y los movimientos del USD/PEN.

Por otro lado, las variantes **Var2_lags_volume_shares** y **Var3_full_temporal**, que incorporan un mayor número de variables, no lograron mejorar el desempeño respecto al baseline. Esto indica que, para el conjunto de datos actual, agregar complejidad no necesariamente produce mejores resultados predictivos.

Asimismo, aunque el baseline presenta un F1 ligeramente superior, la diferencia es pequeña y la métrica principal seleccionada para este proyecto es **PR-AUC**, debido al desbalance existente entre las clases UP y DOWN.

#### Decisión experimental

Se selecciona **Var1_lag1_lag2** como variante candidata para las siguientes etapas del proyecto debido a que:

- Obtiene el mayor PR-AUC.
- Mantiene una baja complejidad (solo 2 features).
- Presenta menor riesgo de sobreajuste.
- Tiene menor costo computacional.
- Conserva interpretabilidad financiera.

Por lo tanto, la evidencia experimental indica que la incorporación de memoria temporal de corto plazo aporta valor predictivo al modelo.

In [ ]:
# =========================
# Gráfico único: PR-AUC + F1
# =========================

plot_df = week6_results_df.copy()

x = np.arange(len(plot_df))
bar_width = 0.35

plt.figure(figsize=(10, 5))

plt.bar(
    x - bar_width / 2,
    plot_df["pr_auc"],
    width=bar_width,
    label="PR-AUC"
)

plt.bar(
    x + bar_width / 2,
    plot_df["f1_binary"],
    width=bar_width,
    label="F1 Binary"
)

plt.xticks(x, plot_df["model"], rotation=20, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Métrica")
plt.title("Comparación A/B - Métrica central y secundaria")
plt.legend()
plt.grid(axis="y", alpha=0.3)

for i, row in plot_df.iterrows():
    plt.text(i - bar_width / 2, row["pr_auc"] + 0.02, f"{row['pr_auc']:.2f}", ha="center", fontsize=9)
    plt.text(i + bar_width / 2, row["f1_binary"] + 0.02, f"{row['f1_binary']:.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(FIGS_DIR / "week6_ab_metricas_centrales.png", dpi=150)
plt.show()

# 6. PR Curve

### Precision-Recall Curve (PR Curve)

La curva Precision-Recall se utilizó como herramienta principal de evaluación debido a que el problema presenta desbalance entre las clases UP y DOWN.

En este contexto:

- **Precision** indica qué proporción de las predicciones positivas realizadas por el modelo fueron correctas.
- **Recall** indica qué proporción de los movimientos positivos reales fue capaz de detectar el modelo.

La variante seleccionada (**Var1_lag1_lag2**) obtuvo:

- **PR-AUC (Average Precision): 0.786**

Este valor resume el comportamiento global de la curva y representa la métrica central utilizada en la comparación de variantes.

### Interpretación

La curva muestra que el modelo mantiene niveles de precision relativamente altos a medida que aumenta el recall, lo que indica una capacidad razonable para identificar movimientos positivos del USD/PEN sin generar una cantidad excesiva de falsos positivos.

El valor obtenido de **PR-AUC = 0.786** sugiere que la incorporación de memoria temporal mediante:

- `lag_sent_1`
- `lag_sent_2`

aporta información útil para la generación de señales financieras basadas en sentimiento.

### Consideraciones

La forma irregular observada en la curva se debe principalmente a:

- tamaño reducido del conjunto de prueba,
- naturaleza temporal del dataset,
- desbalance entre clases,
- número limitado de thresholds generados por el modelo.

Para facilitar la visualización se muestra una versión suavizada de la curva; sin embargo, la métrica reportada (**AP = 0.786**) fue calculada utilizando las probabilidades reales producidas por el modelo.

### Conclusión

La variante **Var1_lag1_lag2** presenta el mejor equilibrio entre:

- desempeño predictivo,
- simplicidad del modelo,
- interpretabilidad,
- y costo computacional.

Por esta razón fue seleccionada como la variante recomendada para las siguientes etapas del proyecto.

In [ ]:
# =========================
# PR Curve 
# =========================

order = np.argsort(recall)
recall_sorted = recall[order]
precision_sorted = precision[order]

recall_grid = np.linspace(0, 1, 100)
precision_interp = np.interp(recall_grid, recall_sorted, precision_sorted)

window = 9
kernel = np.ones(window) / window
precision_smooth = np.convolve(precision_interp, kernel, mode="same")

precision_smooth[0] = precision_sorted[0]
precision_smooth[-1] = precision_sorted[-1]
precision_smooth = np.clip(precision_smooth, 0, 1)

plt.figure(figsize=(7, 5))

plt.plot(
    recall_grid,
    precision_smooth,
    linewidth=2.5,
    label=f"PR Curve suavizada visualmente (AP real = {ap:.3f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall Curve - {best_model_name}")
plt.ylim(0, 1.05)
plt.xlim(0, 1.02)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(FIGS_DIR / "week6_pr_curve_best_model_smooth.png", dpi=150)
plt.show()

# 7. Cross Validation temporal formal



Además del split temporal principal utilizado para train/test, se realizó una validación cruzada temporal mediante **TimeSeriesSplit**.

### Objetivo

Evaluar la estabilidad de cada variante en múltiples ventanas temporales y verificar que el desempeño observado no dependa únicamente de un único período de evaluación.

### Configuración

Se utilizaron:

- 5 folds temporales.
- Entrenamiento únicamente con información pasada.
- Evaluación únicamente sobre períodos futuros.
- Misma semilla y mismo protocolo experimental para todas las variantes.

### Observaciones

Durante los dos primeros folds se observó que el conjunto de entrenamiento contenía únicamente una clase (`train_classes = 1`).

Por esta razón:

- Fold 1 → omitido.
- Fold 2 → omitido.

Esto evidencia una limitación real del dataset y confirma que no se forzó artificialmente el entrenamiento.

Los folds válidos fueron:

- Fold 3
- Fold 4
- Fold 5

### Resultados promedio

| Modelo | Accuracy | F1 Binary | PR-AUC |
|----------|----------|----------|----------|
| Baseline_lag1 | 0.476 | 0.543 | **0.821** |
| Var1_lag1_lag2 | 0.500 | 0.556 | **0.800** |
| Var2_lags_volume_shares | 0.476 | 0.577 | **0.776** |
| Var3_full_temporal | 0.500 | 0.606 | **0.764** |

### Interpretación

Los resultados muestran que las métricas presentan cierta variabilidad entre folds, lo cual es esperado debido al tamaño limitado del conjunto de datos y al comportamiento temporal del mercado.

Sin embargo, todas las variantes mantienen valores de PR-AUC superiores a 0.75, indicando que existe señal predictiva útil en las variables de sentimiento financiero.

Asimismo, se observa que:

- Baseline_lag1 obtuvo el mayor PR-AUC promedio.
- Var1_lag1_lag2 mantuvo un desempeño competitivo y estable.
- Var2 y Var3 aumentaron complejidad sin generar mejoras consistentes en PR-AUC.

### Conclusión

La validación temporal confirma que el desempeño observado en el holdout principal no es producto de un único período específico.

Además, el uso de TimeSeriesSplit permite reducir el riesgo de sobreajuste y fortalece la validez metodológica del experimento.

In [ ]:
# =========================
# Cross Validation temporal formal
# =========================

tscv = TimeSeriesSplit(n_splits=5)

cv_rows = []

for exp in experiments_week6:
    model_name = exp["model"]
    features = exp["features"]

    X = df_model[features].copy()
    y = df_model["target_up"].astype(int).copy()

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
        X_tr = X.iloc[train_idx]
        X_te = X.iloc[test_idx]
        y_tr = y.iloc[train_idx]
        y_te = y.iloc[test_idx]

        train_start = df_model.iloc[train_idx]["date"].min()
        train_end = df_model.iloc[train_idx]["date"].max()
        test_start = df_model.iloc[test_idx]["date"].min()
        test_end = df_model.iloc[test_idx]["date"].max()

        if y_tr.nunique() < 2:
            cv_rows.append({
                "model": model_name,
                "fold": fold,
                "status": "skipped_one_class_train",
                "accuracy": np.nan,
                "f1_binary": np.nan,
                "precision": np.nan,
                "recall": np.nan,
                "pr_auc": np.nan,
                "elapsed_seconds": np.nan,
                "n_train": len(y_tr),
                "n_test": len(y_te),
                "train_classes": y_tr.nunique(),
                "test_classes": y_te.nunique(),
                "train_start": train_start,
                "train_end": train_end,
                "test_start": test_start,
                "test_end": test_end
            })
            continue

        model = make_logreg()

        start = time.time()
        model.fit(X_tr, y_tr)
        elapsed = time.time() - start

        y_pred = model.predict(X_te)
        y_score = model.predict_proba(X_te)[:, 1]

        pr_auc_value = np.nan
        if y_te.nunique() >= 2:
            pr_auc_value = average_precision_score(y_te, y_score)

        cv_rows.append({
            "model": model_name,
            "fold": fold,
            "status": "ok",
            "accuracy": accuracy_score(y_te, y_pred),
            "f1_binary": f1_score(y_te, y_pred, zero_division=0),
            "precision": precision_score(y_te, y_pred, zero_division=0),
            "recall": recall_score(y_te, y_pred, zero_division=0),
            "pr_auc": pr_auc_value,
            "elapsed_seconds": elapsed,
            "n_train": len(y_tr),
            "n_test": len(y_te),
            "train_classes": y_tr.nunique(),
            "test_classes": y_te.nunique(),
            "train_start": train_start,
            "train_end": train_end,
            "test_start": test_start,
            "test_end": test_end
        })

cv_week6_df = pd.DataFrame(cv_rows)
display(cv_week6_df)

cv_valid = cv_week6_df[cv_week6_df["status"] == "ok"].copy()

cv_week6_summary = (
    cv_valid
    .groupby("model")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        f1_binary_mean=("f1_binary", "mean"),
        f1_binary_std=("f1_binary", "std"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        elapsed_mean=("elapsed_seconds", "mean"),
        valid_folds=("fold", "count")
    )
    .reset_index()
)

display(cv_week6_summary)

cv_week6_df.to_csv(
    LOGS_DIR / "week6_cross_validation_folds.csv",
    index=False,
    encoding="utf-8-sig"
)

cv_week6_summary.to_csv(
    LOGS_DIR / "week6_cross_validation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# 8. Gráfico Cross Validation



Con el objetivo de evaluar la estabilidad de los modelos en diferentes períodos del tiempo, se aplicó una validación cruzada temporal mediante **TimeSeriesSplit**.

Cada fold fue construido respetando el orden cronológico de los datos:

- Entrenamiento con información histórica.
- Evaluación sobre períodos futuros.
- Sin mezclar observaciones del futuro en el entrenamiento.

Los folds que contenían únicamente una clase en el conjunto de entrenamiento fueron descartados automáticamente, ya que no permiten entrenar un clasificador binario.

### Resultados promedio

| Modelo | PR-AUC Promedio | F1 Promedio |
|----------|----------|----------|
| Baseline_lag1 | **0.821 ± 0.077** | 0.543 ± 0.078 |
| Var1_lag1_lag2 | 0.800 ± 0.080 | 0.556 ± 0.078 |
| Var2_lags_volume_shares | 0.776 ± 0.039 | 0.577 ± 0.119 |
| Var3_full_temporal | 0.764 ± 0.063 | **0.606 ± 0.087** |

### Interpretación

La validación cruzada muestra que todas las variantes mantienen valores de PR-AUC superiores a 0.75, lo que indica que las variables de sentimiento contienen información predictiva útil para estimar movimientos futuros del USD/PEN.

El modelo **Baseline_lag1** obtuvo el mayor PR-AUC promedio (0.821), mientras que **Var1_lag1_lag2** alcanzó un valor muy cercano (0.800).

Por otro lado, las variantes con mayor número de variables (Var2 y Var3) no lograron mejorar la métrica principal PR-AUC, aunque sí mostraron ligeras mejoras en F1.

### Relación con el Holdout Final

En la evaluación Holdout realizada anteriormente:

- Var1_lag1_lag2 obtuvo el mejor PR-AUC (0.786).

Sin embargo, en la validación cruzada temporal:

- Baseline_lag1 obtuvo el mejor PR-AUC promedio (0.821).

Esto sugiere que la mejora observada en Var1 depende parcialmente del período evaluado y que todavía se requiere una mayor cantidad de datos históricos para confirmar estadísticamente su superioridad.

### Conclusión

La validación temporal confirma que los resultados observados son consistentes en distintos períodos de mercado y reduce el riesgo de sobreajuste.

Actualmente, la evidencia experimental indica que:

- Baseline_lag1 es la alternativa más estable.
- Var1_lag1_lag2 es la alternativa más prometedora para futuras iteraciones.
- Var2 y Var3 incrementan complejidad sin generar mejoras claras en la métrica principal.

In [ ]:
# =========================
# Gráfico Cross Validation
# =========================



import numpy as np
import matplotlib.pyplot as plt

# Usar resumen generado previamente:
# cv_week6_summary

plot_cv = cv_week6_summary.copy()

# Orden sugerido
order_models = [
    "Baseline_lag1",
    "Var1_lag1_lag2",
    "Var2_lags_volume_shares",
    "Var3_full_temporal"
]

plot_cv["model"] = pd.Categorical(
    plot_cv["model"],
    categories=order_models,
    ordered=True
)

plot_cv = plot_cv.sort_values("model").reset_index(drop=True)

x = np.arange(len(plot_cv))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

bars_pr = ax.bar(
    x - bar_width / 2,
    plot_cv["pr_auc_mean"],
    yerr=plot_cv["pr_auc_std"],
    width=bar_width,
    label="PR-AUC mean ± std",
    capsize=5
)

bars_f1 = ax.bar(
    x + bar_width / 2,
    plot_cv["f1_binary_mean"],
    yerr=plot_cv["f1_binary_std"],
    width=bar_width,
    label="F1 mean ± std",
    capsize=5
)

# Etiquetas encima de barras PR-AUC
for i, bar in enumerate(bars_pr):
    height = bar.get_height()
    std = plot_cv.loc[i, "pr_auc_std"]

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.025,
        f"{height:.3f}\n±{std:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# Etiquetas encima de barras F1
for i, bar in enumerate(bars_f1):
    height = bar.get_height()
    std = plot_cv.loc[i, "f1_binary_std"]

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.025,
        f"{height:.3f}\n±{std:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# Marcar mejor PR-AUC promedio
best_idx = plot_cv["pr_auc_mean"].idxmax()
best_x = x[best_idx] - bar_width / 2
best_y = plot_cv.loc[best_idx, "pr_auc_mean"]

ax.annotate(
    "Mejor PR-AUC promedio",
    xy=(best_x, best_y),
    xytext=(best_x, min(best_y + 0.15, 1.02)),
    arrowprops=dict(arrowstyle="->", linewidth=1.2),
    ha="center",
    fontsize=9
)

ax.set_xticks(x)
ax.set_xticklabels(plot_cv["model"], rotation=20, ha="right")
ax.set_ylim(0, 1.12)
ax.set_ylabel("Métrica promedio")
ax.set_title("Cross Validation Temporal - TimeSeriesSplit")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Nota metodológica
fig.text(
    0.5,
    -0.03,
    "Nota: métricas promedio calculadas sobre folds temporales válidos. "
    "Folds con una sola clase en train fueron omitidos.",
    ha="center",
    fontsize=9
)

plt.tight_layout()

plt.savefig(
    FIGS_DIR / "week6_cross_validation_summary_annotated.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

# 9. Decisión experimental final


Con el objetivo de seleccionar la variante más adecuada para futuras iteraciones, se comparó cada experimento respecto al baseline utilizando la métrica principal PR-AUC.

### Resultados

| Modelo | PR-AUC | Δ PR-AUC vs Baseline | Decisión |
|----------|----------|----------|----------|
| Baseline_lag1 | 0.777 | 0.000 | Referencia |
| Var1_lag1_lag2 | 0.786 | +0.009 | Evaluar |
| Var2_lags_volume_shares | 0.746 | -0.031 | Descartar |
| Var3_full_temporal | 0.741 | -0.036 | Descartar |

### Interpretación

La variante Var1_lag1_lag2 obtuvo el mejor resultado en la evaluación holdout, alcanzando un PR-AUC de 0.786 frente a 0.777 del baseline.

La mejora observada es positiva pero relativamente pequeña (+0.009), por lo que aún no constituye evidencia suficiente para justificar una adopción definitiva.

Por otro lado, las variantes Var2 y Var3 incrementaron la complejidad del modelo mediante la incorporación de nuevas variables temporales y de volumen, pero no lograron superar el desempeño del baseline.

### Decisión

Se concluye que:

- Baseline_lag1 continúa siendo la referencia principal.
- Var1_lag1_lag2 es la variante más prometedora.
- Var2 y Var3 son descartadas para futuras iteraciones.

La recomendación actual es continuar evaluando Var1 con una mayor cantidad de datos históricos y validaciones adicionales antes de considerar su adopción definitiva.

In [ ]:
# =========================
# Decisión experimental final
# =========================

base = week6_results_df[week6_results_df["model"] == "Baseline_lag1"].iloc[0]

decision_rows = []

for _, row in week6_results_df.iterrows():
    delta_pr_auc = row["pr_auc"] - base["pr_auc"]
    delta_f1 = row["f1_binary"] - base["f1_binary"]

    if row["model"] == "Baseline_lag1":
        decision = "Referencia"
        reason = "Baseline usado como punto de comparación."
    elif delta_pr_auc > 0 and delta_f1 >= 0 and row["elapsed_seconds"] <= base["elapsed_seconds"] * 3:
        decision = "Adoptar"
        reason = "Mejora PR-AUC y F1 con costo/latencia aceptable."
    elif delta_pr_auc > 0:
        decision = "Evaluar"
        reason = "Mejora PR-AUC, pero requiere revisar costo o estabilidad."
    else:
        decision = "Descartar"
        reason = "No mejora la métrica central frente al baseline."

    decision_rows.append({
        "model": row["model"],
        "pr_auc": row["pr_auc"],
        "f1_binary": row["f1_binary"],
        "accuracy": row["accuracy"],
        "elapsed_seconds": row["elapsed_seconds"],
        "delta_pr_auc_vs_baseline": delta_pr_auc,
        "delta_f1_vs_baseline": delta_f1,
        "decision": decision,
        "reason": reason
    })

week6_decision_df = pd.DataFrame(decision_rows)
display(week6_decision_df)

week6_decision_df.to_csv(
    LOGS_DIR / "week6_decision_experimental.csv",
    index=False,
    encoding="utf-8-sig"
)